# Module 02: Pandas for Machine Learning
## Notebook 03: Data Cleaning and Missing Value Imputation

Real-world machine learning data is messy, incomplete, and corrupted. Machine learning models (especially gradient descent and neural networks) will throw errors or produce biased predictions if presented with missing (`NaN`) values or unstandardized strings.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Detect and quantify missing values using `.isna()`, `.notna()`, and missingness ratios.
2. Select appropriate imputation strategies (mean, median, mode, forward-fill) for different data distributions.
3. Construct **Missingness Indicator** features to preserve predictive signal.
4. Detect and eliminate duplicate observations.
5. Standardize corrupted string features using Pandas vectorized `.str` methods.

In [1]:
import pandas as pd
import numpy as np

print(f"Pandas version: {pd.__version__}")

Pandas version: 3.0.6


### 1. Detecting and Quantifying Missing Values

In Pandas, missing values are represented as `np.nan` (or `pd.NA`).
Key inspection methods:
- `df.isna().sum()`: Absolute count of missing entries per column.
- `df.isna().mean() * 100`: Percentage of missing data per feature.

In [2]:
# Create a simulated noisy ML dataset
raw_data = {
    'Applicant_ID': [101, 102, 103, 104, 105, 106, 107, 108],
    'Age': [25.0, np.nan, 45.0, 31.0, np.nan, 52.0, 29.0, 38.0],
    'Salary': [50000.0, 62000.0, np.nan, 75000.0, 48000.0, 150000.0, np.nan, 70000.0],
    'Education': ['BSc', 'MSc', 'PhD', np.nan, 'BSc', 'MSc', 'BSc', np.nan],
    'City': [' New York ', 'los angeles', 'NEW YORK', 'Chicago', 'Chicago ', 'Houston', 'houston', 'New York']
}

df = pd.DataFrame(raw_data)
print("Raw Dirty Dataset:\n", df)

print("\n--- Missing Value Audit ---")
missing_summary = pd.DataFrame({
    'Missing_Count': df.isna().sum(),
    'Missing_Percent': (df.isna().mean() * 100).round(2)
})
print(missing_summary)

Raw Dirty Dataset:
    Applicant_ID   Age    Salary Education         City
0           101  25.0   50000.0       BSc    New York 
1           102   NaN   62000.0       MSc  los angeles
2           103  45.0       NaN       PhD     NEW YORK
3           104  31.0   75000.0       NaN      Chicago
4           105   NaN   48000.0       BSc     Chicago 
5           106  52.0  150000.0       MSc      Houston
6           107  29.0       NaN       BSc      houston
7           108  38.0   70000.0       NaN     New York

--- Missing Value Audit ---
              Missing_Count  Missing_Percent
Applicant_ID              0              0.0
Age                       2             25.0
Salary                    2             25.0
Education                 2             25.0
City                      0              0.0


---
### 2. Dropping vs. Imputing Missing Values

- **Dropping (`.dropna()`)**: Reasonable when:
  - Missingness is minimal (< 2-3% of rows).
  - A feature is almost entirely missing (> 70-80% missing) and unrecoverable.
- **Imputing (`.fillna()`)**: Essential when dropping data introduces sampling bias or destroys sample size.

In [3]:
# Example of dropna with threshold: keep rows that have at least 4 non-null values
df_dropped = df.dropna(thresh=4)
print(f"Rows before drop: {len(df)} | Rows after drop: {len(df_dropped)}")

Rows before drop: 8 | Rows after drop: 8


---
### 3. Statistical Imputation Strategies

| Feature Type | Best Practice Imputer | Rationale |
|---|---|---|
| Normally distributed numeric | **Mean** | Center of symmetric distribution |
| Skewed numeric / Outliers present | **Median** | Robust to extreme outliers |
| Categorical feature | **Mode** (most frequent) | Plausible category assignment |
| Time Series | **ffill** / **bfill** | Temporal continuity |

> **ML Pro-Tip: Missingness Indicators**
> Sometimes the fact that a value was missing is itself highly predictive (e.g., patient refused to report income).
> Create a binary flag `column_is_missing` before imputing!

In [4]:
df_clean = df.copy()

# 1. Add missingness indicator before imputation
df_clean['Salary_Was_Missing'] = df_clean['Salary'].isna().astype(int)

# 2. Impute Age with Mean
age_mean = df_clean['Age'].mean()
df_clean['Age'] = df_clean['Age'].fillna(age_mean)

# 3. Impute Salary with Median (robust to the 150,000 outlier)
salary_median = df_clean['Salary'].median()
df_clean['Salary'] = df_clean['Salary'].fillna(salary_median)

# 4. Impute Categorical Education with Mode
education_mode = df_clean['Education'].mode()[0]
df_clean['Education'] = df_clean['Education'].fillna(education_mode)

print("Dataset after Imputation:\n", df_clean)

Dataset after Imputation:
    Applicant_ID        Age    Salary Education         City  \
0           101  25.000000   50000.0       BSc    New York    
1           102  36.666667   62000.0       MSc  los angeles   
2           103  45.000000   66000.0       PhD     NEW YORK   
3           104  31.000000   75000.0       BSc      Chicago   
4           105  36.666667   48000.0       BSc     Chicago    
5           106  52.000000  150000.0       MSc      Houston   
6           107  29.000000   66000.0       BSc      houston   
7           108  38.000000   70000.0       BSc     New York   

   Salary_Was_Missing  
0                   0  
1                   0  
2                   1  
3                   0  
4                   0  
5                   0  
6                   1  
7                   0  


---
### 4. Detecting and Removing Duplicates

Duplicate records falsely inflate validation scores and bias gradient updates toward repeated samples.
- `df.duplicated()`: Identifies identical rows.
- `df.drop_duplicates()`: Removes duplicates.

In [5]:
# Simulate a dataset with duplicate entries
records = pd.DataFrame({
    'User': ['Alice', 'Bob', 'Alice', 'Charlie', 'Bob'],
    'Action': ['Click', 'Buy', 'Click', 'Scroll', 'Buy']
})

print("Original Records:\n", records)
print("\nDuplicated mask:\n", records.duplicated())

clean_records = records.drop_duplicates()
print("\nDeduplicated Records:\n", clean_records)

Original Records:
       User  Action
0    Alice   Click
1      Bob     Buy
2    Alice   Click
3  Charlie  Scroll
4      Bob     Buy

Duplicated mask:
 0    False
1    False
2     True
3    False
4     True
dtype: bool

Deduplicated Records:
       User  Action
0    Alice   Click
1      Bob     Buy
3  Charlie  Scroll


---
### 5. Cleaning String Features with `.str`

In the raw dataset, the `City` column contains casing mismatches (`'los angeles'` vs `'Los Angeles'`) and whitespace (`' New York '` vs `'NEW YORK'`).
To ML algorithms, these appear as completely different categories!

In [6]:
print("Raw City values before cleaning:\n", df_clean['City'].value_counts())

# 1. Strip leading and trailing whitespace
df_clean['City'] = df_clean['City'].str.strip()

# 2. Standardize title case (e.g. 'los angeles' -> 'Los Angeles')
df_clean['City'] = df_clean['City'].str.title()

print("\nCleaned City values after .strip() and .title():\n", df_clean['City'].value_counts())

Raw City values before cleaning:
 City
 New York      1
los angeles    1
NEW YORK       1
Chicago        1
Chicago        1
Houston        1
houston        1
New York       1
Name: count, dtype: int64

Cleaned City values after .strip() and .title():
 City
New York       3
Chicago        2
Houston        2
Los Angeles    1
Name: count, dtype: int64


### Summary & Next Steps
In this notebook, you mastered:
- Calculating missing data rates across features.
- Imputing with mean, median, and mode while preserving missingness signals.
- Dropping redundant duplicate observations.
- Standardizing messy categorical strings using `.str` methods.

**Next Notebook:** `04_aggregations_grouping_and_pivot_tables.ipynb` — Master `groupby()`, multi-column aggregations, `.transform()`, and pivot tables.